# 03 — Fine-tune DeepSeek-OCR (LoRA)
One notebook, used identically for all three fine-tuning runs. Set `MODE` in the cell below and run top to bottom. Run it three times: once per mode. Nothing else changes between runs.

| MODE | Training data | Steps |
|---|---|---|
| `raw1x` | 8,500 raw images, once each | ~1,062 |
| `rawrepeat` | same raw images, cycled to 100K | 12,500 |
| `aug100k` | 100K sampled from the augmented pool | 12,500 |

This version does not merge the LoRA adapter into the base model — a current Unsloth bug silently fails to merge DeepSeek-OCR (it writes out roughly the adapter's own size instead of a full checkpoint, with no error). Notebook 04 loads the base model plus the adapter directly, which sidesteps the bug and needs no merged checkpoint. The adapter produced here is the only output you need.

In [ ]:
MODE = "rawrepeat"     # "raw1x" | "rawrepeat" | "aug100k"  <- set this, then Run All
assert MODE in ("raw1x", "rawrepeat", "aug100k")
USE_VAL = True      # tracks validation loss during training; set False only if it errors on your setup

## GPU check
No `pip install` here. On Windows, installing inside the notebook can silently replace a CUDA build of PyTorch with a CPU-only one. Install packages once from a terminal, before opening Jupyter, then just verify here.

In [ ]:
import torch
print("torch:", torch.__version__, "| CUDA build:", torch.version.cuda, "| GPU visible:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU visible to PyTorch. Reinstall a CUDA build from a terminal, e.g. cu128 for an RTX 50-series card, then restart the kernel."
print("GPU:", torch.cuda.get_device_name(0))

## Load model and apply LoRA

In [ ]:
import os, sys, json, time
os.environ["UNSLOTH_WARN_UNINITIALIZED"] = "0"
import config as C
sys.path.insert(0, str(C.DEEPSEEK_SRC))
import pandas as pd
from unsloth import FastVisionModel, is_bf16_supported
from transformers import AutoModel, Trainer, TrainingArguments, TrainerCallback
from datasets import Dataset, Features, Image, Value

if not C.BASE_MODEL.exists():
    from huggingface_hub import snapshot_download   # set HF_TOKEN in your environment first if the download needs it
    snapshot_download("unsloth/DeepSeek-OCR", local_dir=str(C.BASE_MODEL))

model, tokenizer = FastVisionModel.from_pretrained(
    model_name=str(C.BASE_MODEL), load_in_4bit=False, auto_model=AutoModel, trust_remote_code=True,
    unsloth_force_compile=True, use_gradient_checkpointing="unsloth")

model = FastVisionModel.get_peft_model(
    model, target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=3407, use_rslora=False, loftq_config=None)

n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_all = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {n_tr:,} / {n_all:,} ({100*n_tr/n_all:.2f}%)")
assert n_tr == 77_509_632, "Trainable parameter count mismatch versus the expected LoRA configuration"

## Load the training data for the selected mode

In [ ]:
def load_ds(name):
    d = pd.read_csv(C.SUBSET_DIR / f"{name}.csv")
    ds = Dataset.from_pandas(d, features=Features({"image": Image(), "text": Value("string")}), preserve_index=False)
    def conv(s):
        return {"messages": [{"role": "<|User|>", "content": C.PROMPT, "images": [s["image"]]},
                             {"role": "<|Assistant|>", "content": s["text"]}]}
    return ds.map(conv)

train_ds = load_ds(MODE)
val_ds = load_ds("val_loss") if USE_VAL else None
print(MODE, "| train samples:", len(train_ds), "| val samples:", len(val_ds) if val_ds else 0)

## Data collator (`ocr_collator.py`)
Builds the image tokens and masks the prompt so the loss is computed on the answer only. Identical for all three modes.

In [ ]:
%%writefile ocr_collator.py
import math, io
from typing import Any
import torch
from PIL import Image, ImageOps
from torch.nn.utils.rnn import pad_sequence
from deepseek_ocr.modeling_deepseekocr import text_encode, BasicImageTransform, dynamic_preprocess

class DeepSeekOCRDataCollator:
    def __init__(self, tokenizer, model, image_size=640, base_size=1024, crop_mode=True, train_on_responses_only=True):
        self.tokenizer, self.model = tokenizer, model
        self.image_size, self.base_size, self.crop_mode = image_size, base_size, crop_mode
        self.image_token_id = 128815
        self.dtype = model.dtype
        self.train_on_responses_only = train_on_responses_only
        self.image_transform = BasicImageTransform(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5), normalize=True)
        self.patch_size, self.downsample_ratio = 16, 4
        self.bos_id = getattr(tokenizer, "bos_token_id", None) or 0

    def deserialize_image(self, d):
        if isinstance(d, Image.Image): return d.convert("RGB")
        if isinstance(d, dict):
            if d.get("path") is not None: return Image.open(d["path"]).convert("RGB")
            if d.get("bytes") is not None: return Image.open(io.BytesIO(d["bytes"])).convert("RGB")
        raise ValueError(f"Unsupported image format: {type(d)}")

    def process_image(self, image):
        images_list, images_crop_list, spatial = [], [], []
        if self.crop_mode:
            if image.size[0] <= 640 and image.size[1] <= 640:
                crop_ratio, crops = (1, 1), []
            else:
                crops, crop_ratio = dynamic_preprocess(image, min_num=2, max_num=9, image_size=self.image_size, use_thumbnail=False)
            gv = ImageOps.pad(image, (self.base_size, self.base_size), color=tuple(int(x * 255) for x in self.image_transform.mean))
            images_list.append(self.image_transform(gv).to(self.dtype))
            wc, hc = crop_ratio
            spatial.append([wc, hc])
            if wc > 1 or hc > 1:
                for ci in crops: images_crop_list.append(self.image_transform(ci).to(self.dtype))
            nq = math.ceil((self.image_size // self.patch_size) / self.downsample_ratio)
            nqb = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)
            tok = ([self.image_token_id] * nqb + [self.image_token_id]) * nqb + [self.image_token_id]
            if wc > 1 or hc > 1:
                tok += ([self.image_token_id] * (nq * wc) + [self.image_token_id]) * (nq * hc)
        else:
            spatial.append([1, 1])
            if self.base_size <= 640:
                r = image.resize((self.base_size, self.base_size), Image.LANCZOS)
                images_list.append(self.image_transform(r).to(self.dtype))
            else:
                gv = ImageOps.pad(image, (self.base_size, self.base_size), color=tuple(int(x * 255) for x in self.image_transform.mean))
                images_list.append(self.image_transform(gv).to(self.dtype))
            nq = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)
            tok = ([self.image_token_id] * nq + [self.image_token_id]) * nq + [self.image_token_id]
        return images_list, images_crop_list, spatial, tok

    def process_single_sample(self, messages):
        images = [self.deserialize_image(i) for m in messages if m.get("images") for i in m["images"] if i is not None]
        if not images: raise ValueError("No images found in sample.")
        ids, seqmask = [self.bos_id], [False]
        il, icl, isc = [], [], []
        prompt_count, started, k = -1, False, 0
        for m in messages:
            role, content = m["role"], m["content"]
            if role == "<|Assistant|>":
                if not started:
                    prompt_count, started = len(ids), True
                content = f"{content.strip()} {self.tokenizer.eos_token}"
            parts = content.split("<image>")
            for i, p in enumerate(parts):
                t = text_encode(self.tokenizer, p, bos=False, eos=False)
                ids.extend(t); seqmask.extend([False] * len(t))
                if i < len(parts) - 1:
                    a, b, s, tok = self.process_image(images[k])
                    il.extend(a); icl.extend(b); isc.extend(s)
                    ids.extend(tok); seqmask.extend([True] * len(tok)); k += 1
        if not started: prompt_count = len(ids)
        crop = torch.stack(icl, 0) if icl else torch.zeros((1, 3, self.base_size, self.base_size), dtype=self.dtype)
        return {"input_ids": torch.tensor(ids, dtype=torch.long), "images_seq_mask": torch.tensor(seqmask, dtype=torch.bool),
                "images_ori": torch.stack(il, 0), "images_crop": crop,
                "images_spatial_crop": torch.tensor(isc, dtype=torch.long), "prompt_token_count": prompt_count}

    def __call__(self, features):
        batch = []
        for f in features:
            try: batch.append(self.process_single_sample(f["messages"]))
            except Exception as e: print(f"Error processing sample: {e}")
        if not batch: raise ValueError("No valid samples in batch")
        pad = self.tokenizer.pad_token_id
        input_ids = pad_sequence([b["input_ids"] for b in batch], batch_first=True, padding_value=pad)
        seqmask = pad_sequence([b["images_seq_mask"] for b in batch], batch_first=True, padding_value=False)
        labels = input_ids.clone()
        labels[labels == pad] = -100
        labels[seqmask] = -100
        if self.train_on_responses_only:
            for i, b in enumerate(batch):
                if b["prompt_token_count"] > 0: labels[i, :b["prompt_token_count"]] = -100
        return {"input_ids": input_ids, "attention_mask": (input_ids != pad).long(), "labels": labels,
                "images": [(b["images_crop"], b["images_ori"]) for b in batch], "images_seq_mask": seqmask,
                "images_spatial_crop": torch.cat([b["images_spatial_crop"] for b in batch], 0)}

## Trainer, with live loss logging built in
Every logged step is appended to `loss_<mode>_live.csv` immediately, so nothing is lost if a run is interrupted. Checkpoints every 2,000 steps are a second fallback.

In [ ]:
from ocr_collator import DeepSeekOCRDataCollator

class LiveLossLogger(TrainerCallback):
    def __init__(self, path):
        self.path = path
        self.path.write_text("step,loss,eval_loss,learning_rate\n")
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None: return
        with open(self.path, "a") as f:
            f.write(f"{state.global_step},{logs.get('loss','')},{logs.get('eval_loss','')},{logs.get('learning_rate','')}\n")

FastVisionModel.for_training(model)
collator = DeepSeekOCRDataCollator(tokenizer=tokenizer, model=model, image_size=C.IMAGE_SIZE,
                                   base_size=C.BASE_SIZE, crop_mode=False, train_on_responses_only=True)

eval_every = 200 if MODE == "raw1x" else 1000
args = TrainingArguments(
    per_device_train_batch_size=2, gradient_accumulation_steps=4, warmup_steps=5, num_train_epochs=1,
    learning_rate=2e-4, logging_steps=1, optim="adamw_8bit", weight_decay=0.001, lr_scheduler_type="linear",
    seed=3407, fp16=not is_bf16_supported(), bf16=is_bf16_supported(),
    output_dir=str(C.MODELS_DIR / f"{MODE}_checkpoints"), save_strategy="steps", save_steps=2000, save_total_limit=2,
    report_to="none", dataloader_num_workers=0, remove_unused_columns=False,
    **({"eval_strategy": "steps", "eval_steps": eval_every, "per_device_eval_batch_size": 2} if USE_VAL else {}))
trainer = Trainer(model=model, tokenizer=tokenizer, data_collator=collator,
                  train_dataset=train_ds, eval_dataset=val_ds, args=args)
trainer.add_callback(LiveLossLogger(C.RESULTS_DIR / f"loss_{MODE}_live.csv"))

print(f"{MODE}: {len(train_ds)//8:,} optimizer steps")
print("Live loss ->", C.RESULTS_DIR / f"loss_{MODE}_live.csv", "(safe to open and plot mid-training)")

## Train

In [ ]:
t0 = time.time()
stats = trainer.train()
minutes = (time.time() - t0) / 60
print(f"Training finished in {minutes:.1f} min")

# Consolidated loss history, written once training completes (the _live.csv above already has this if it didn't)
log = trainer.state.log_history
json.dump(log, open(C.RESULTS_DIR / f"loss_{MODE}.json", "w"), indent=1)
rows = [{"step": e["step"], "loss": e.get("loss"), "eval_loss": e.get("eval_loss"), "learning_rate": e.get("learning_rate")}
        for e in log if ("loss" in e or "eval_loss" in e)]
pd.DataFrame(rows).to_csv(C.RESULTS_DIR / f"loss_{MODE}.csv", index=False)
json.dump({"mode": MODE, "train_samples": len(train_ds), "steps": trainer.state.global_step, "minutes": minutes,
           "gpu": torch.cuda.get_device_name(0), "peak_gpu_gb": torch.cuda.max_memory_reserved() / 1024**3},
          open(C.RESULTS_DIR / f"train_info_{MODE}.json", "w"), indent=1)
print("Saved loss_%s.csv and train_info_%s.json" % (MODE, MODE))

## Save the adapter
This is the only model artifact this notebook produces. Notebook 04 loads the base model plus this adapter directly — no merge step needed.

In [ ]:
adapter_dir = C.MODELS_DIR / f"{MODE}_lora"
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print("Adapter saved to", adapter_dir)
print("\nDone. To run the next mode: change MODE in the second cell and use Kernel -> Restart and Run All.")